[ Known Bad Account ] ──(Shares Device)──► [ Target Account (Looks Clean) ]
         │                                               │
         └─────────────(Transfers Money)─────────────────┘
                                 │
                   GNN Message Passing (SAGEConv)
                                 │
                                 ▼
              Target Account Flagged as High Risk!
              

In [ ]:
# Install PyTorch Geometric in Colab (takes ~30 seconds)
!pip install -q torch-geometric

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

# 1. Node Features (8 Nodes, 2 features each: [transaction_volume, failed_logins])
# Node 0, 1: Known Normal (Class 0)
# Node 6, 7: Known Malicious (Class 1)
# Node 3, 4: Target nodes with clean-looking stats, but connected to fraudsters!
x = torch.tensor([
    [10.0, 0.0],  # Node 0: Normal
    [15.0, 1.0],  # Node 1: Normal
    [12.0, 0.0],  # Node 2: Normal
    [11.0, 0.0],  # Node 3: Clean stats, but connects to bad actor 6
    [14.0, 0.0],  # Node 4: Clean stats, but connects to bad actor 7
    [18.0, 1.0],  # Node 5: Normal
    [95.0, 12.0], # Node 6: Known Fraud (High transactions & failed logins)
    [85.0, 15.0], # Node 7: Known Fraud
], dtype=torch.float)

# 2. Graph Connectivity (COO format: [source_nodes, target_nodes])
# Undirected edges connecting accounts
edge_index = torch.tensor([
    [0, 1, 1, 2, 2, 0, 3, 6, 4, 7, 6, 7, 3, 4],
    [1, 0, 2, 1, 0, 2, 6, 3, 7, 4, 7, 6, 4, 3]
], dtype=torch.long)

# 3. Ground Truth Labels: 0 = Normal, 1 = Fraud / High Risk
y = torch.tensor([0, 0, 0, 1, 1, 0, 1, 1], dtype=torch.long)

# 4. Train / Test Masks (Train on known nodes 0, 1, 6, 7; Test on hidden nodes 3, 4)
train_mask = torch.tensor([True, True, False, False, False, False, True, True], dtype=torch.bool)
test_mask = torch.tensor([False, False, True, True, True, True, False, False], dtype=torch.bool)

# Build PyG Data Object
graph_data = Data(x=x, edge_index=edge_index, y=y, train_mask=train_mask, test_mask=test_mask)
print(f"Graph loaded successfully with {graph_data.num_nodes} nodes and {graph_data.num_edges} edges!")

In [ ]:
class FraudGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(FraudGNN, self).__init__()
        # Layer 1: Aggregates 1-hop neighbor features
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='mean')
        # Layer 2: Aggregates 2-hop neighbor features
        self.conv2 = SAGEConv(hidden_channels, out_channels, aggr='mean')

    def forward(self, x, edge_index):
        # 1. First neighborhood message passing
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.2, training=self.training)

        # 2. Second neighborhood message passing
        out = self.conv2(h, edge_index)
        return out, h  # Returns predictions + learned GNN embeddings

# Initialize GNN: 2 input features -> 16 hidden dim -> 2 output classes (Normal/Fraud)
model = FraudGNN(in_channels=2, hidden_channels=16, out_channels=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

print(model)

In [ ]:
model.train()
for epoch in range(1, 101):
    optimizer.zero_grad()
    out, embeddings = model(graph_data.x, graph_data.edge_index)

    # Compute loss only on known training nodes
    loss = criterion(out[graph_data.train_mask], graph_data.y[graph_data.train_mask])
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")

print("\n Training Complete!")

In [ ]:
model.eval()
with torch.no_grad():
    out, gnn_embeddings = model(graph_data.x, graph_data.edge_index)
    probabilities = F.softmax(out, dim=1)
    predictions = out.argmax(dim=1)

print("\n===== NODE CLASSIFICATION RESULTS =====")
for node_id in range(graph_data.num_nodes):
    pred_label = "0 FRAUD / HIGH RISK" if predictions[node_id] == 1 else "1 NORMAL"
    true_label = "Fraud" if graph_data.y[node_id] == 1 else "Normal"
    fraud_prob = probabilities[node_id][1].item() * 100

    status = "(Hidden Test Node)" if graph_data.test_mask[node_id] else "(Training Node)"
    print(f"Node {node_id} {status:19s} | True: {true_label:6s} | Predicted: {pred_label} (Fraud Prob: {fraud_prob:.1f}%)")